# Attemps at automatic scansion of poetic lines

## 1. Library import

In [1]:
import torch.nn as nn
import torch
import random
from syllabipy.sonoripy import SonoriPy
import nltk
from nltk.corpus import cmudict
import re
import pandas as pd
import random
import numpy as np
from tqdm import tqdm
import torch.optim as optim

## 2. Syllabifiers testing

Hyphenator

In [2]:
from hyphen import Hyphenator
h_en = Hyphenator('en_US')


In [3]:
h_en.syllables('bastard') # a stupid test

['bas', 'tard']

In [11]:
def syllabifier(word):
    # Hyphenator is meant to hyphen words which is not possible with monosyllabic words. Hence this function.
    sylls = h_en.syllables(word)
    if sylls == [] :
        return [word]
    else :
        return sylls

In [5]:
syllabifier("the")

['the']

In [9]:
# preprocessing functions and variables
nltk.download('punkt', quiet=True)

PUNCT_RE = re.compile(r'[^\w\s]+$')

def is_punct(string):  
    return PUNCT_RE.match(string) is not None

SonoriPy (seehttps://github.com/henchc/syllabipy). As we can see with a few examples, this algorithm is not foolproof...

In [7]:
SonoriPy("justification")

['jus', 'ti', 'fi', 'ca', 'tion']

In [6]:
SonoriPy("doing") # a problem

['doing']

For the tests :

In [12]:
line = 	"What was he doing the great god Pan" 

## 3. First strategy : handcrafted rule-based algorithm

In [14]:
def stress_rules(token, syllables, tags):

    if len(syllables) == 1 :
        stress_per_syll = [1] 
        # By default, monosyllables are stressed, of course; but in poetry, we get the impression that quite often they can be unstressed.
        # If they are function words like "the", "of", etc. We were thinking of integrating this into another feature (POS-TAG for example) and here
        # simply reproducing the rules of normal lexical stress, which, being often mishandled by poets, cannot constitute a sufficient feature
        # and deterministic.
    # If a verb
    elif tags[token] in ["VB","VBD","VBG","VBN","BP", "VBZ"] :
        # 2 syllables -> stress on the second, almost invariably 
        if len(syllables) == 2 :
            if token[-2:] in ["ry", "er"] : # exceptions
                stress_per_syll = [1, 0]
            else : 
                stress_per_syll = [0, 1]
        else : # For long verbs, there is, to our knowledge, no foolproof technique: so it's random, but we could also say that we fall back on the following cases:
            index = random.randrange(0,len(syllables)-1)
            stress_per_syll = [0] * index + [1] + [len(syllables) - (index + 1)] 
    else : # These are the special cases for non-verbs depending on the suffixes and number of syllables.
        if len(syllables) == 2 and (token[-2:] in ["ce", "se"] or token[-3:] in ["eit", "ief"] or token[-4:] == "aint") :
            stress_per_syll = [0, 1]
        # Words with prefixes (verbs as well as nouns): stress on the first syllable of the root (hence 'understand')
        elif (token[-3:] in ["ade", "ese", "eer", "que", "oon", "aire"]) or (token[-2:] in ["ee", "oo"]) or (token[-4:] in[ "ette", "elle"]) or (token[-5:] == "esque"):
            print("ok")
            stress_per_syll = [0] * (len(syllables) - 1) + [1]
        elif (token[-4:] in ["sion", "tion", "able", "cian", "ious", "ient", "iant" "osis", "itis"]) or (token[-2:] in ["ic", "ia"]) or (token[-3:] in ["ial", "ery", "ian", "ish", "ion", "ics", "ing", "ium", "ior", "iar"]) :
            stress_per_syll = [0] * (len(syllables) - 2) + [1, 0]
        elif (token[-2:] in ["ly", "er"]) or (token[-3:] in ["ism", "ist", "ous"]):
            stress_per_syll = [1] + [0] * (len(syllables)-1)
        elif len(syllables) >= 3 and (token[-2:] in ["cy", "ty", "gy", "al"] or token[-3:] in ["ify"]) :
            stress_per_syll = [0] * (len(syllables) - 3) + [1] + [0] *2
        else :
            if len(syllables) == 2 : # Otherwise, in general, two-syllable words have the stress on the first syllable.
                stress_per_syll = [1, 0]
            elif len(syllables) == 3: # those with 3 syllables on the penultimate
                stress_per_syll = stress_per_syll = [0] * (len(syllables) - 2) + [1, 0]
            else : # those with more on the third-to-last
                stress_per_syll = stress_per_syll = [0] * (len(syllables) - 3) + [1] + [0] *2
    return stress_per_syll

Some tests

In [15]:
stress_rules("justification", ["jus", "ti", "fi", "ca","tion"], {"justification" : "NN"})

[0, 0, 0, 1, 0]

In [16]:
stress_rules("chivalrous", ["chi", "val", "rous"], {"chivalrous" : "JJ"})

[1, 0, 0]

## 4. Strategy 2 : lexical stress according to a dictionary of English word pronunciations (the CMU)

For more information on the CMU, see http://www.speech.cs.cmu.edu/cgi-bin/cmudict

In [18]:
# cmu is included in nltk
cmu = cmudict.dict()

# list of english vowels (in Arpabet : https://en.wikipedia.org/wiki/ARPABET)
VOWELS = ['AA','AE','AH','AO','AW','AY','EH','ER','EY','IH','IY','OW','OY','UH','UW']


0 means that there is no accent on the vowel; 1 that it is the main stress accent (the one that interests us); 2 that it is the secondary stress accent etc.

In [19]:
cmu["justification"]

[['JH', 'AH2', 'S', 'T', 'AH0', 'F', 'AH0', 'K', 'EY1', 'SH', 'AH0', 'N']]

Problem: not all the words are there... But there is the option of reverting to our rule system in this case.

Feature-extraction algorithm.

In [20]:
# The input requires a tokenized line of verse and its associated POS tags (in case we revert to the rule-based system).
# The output is a list of tuples of the form (syllable, token, index of the syllable within the token, 1/0 depending on whether the syllable is stressed or unstressed, 1/0 depending on whether the syllable is actually a monosyllabic word or not) for each syllable in the line of verse.
def lexical_stress_tokens(tokens_list, tags) :
    syllables_list = []

    for token in tokens_list:
        #toke_syll = SonoriPy(token) # dividing a word into syllables (rarely perfect, alack)
        toke_syll = syllabifier(token)
        monosyllabic = 1 if len(toke_syll) == 1 else 0 # detection of monosyllabic words

        try : 
            cmu_phones = cmu[token][0] # if the word is in the CMU
        except KeyError:
            stress_per_syll = stress_rules(token=token, syllables=toke_syll, tags=tags) # if not -> rule-based system
            for index, syll in enumerate(toke_syll) :
                syllables_list.append((syll, token, index, stress_per_syll[index], monosyllabic))
            continue 
        
        # if the word is in the CMU
        # The problem then is that while cmu_phones does return the position of the stress in the word, it doesn't break it down into syllables.

        # So you have to try to link the stressed phoneme found by the CMU to the syllable that carries it (in the toke_syll list).
        phonemes_per_syll = [[] for _ in toke_syll] 
        current_syll = 0
        vowel_count = 0
        for ph in cmu_phones: 
            phonemes_per_syll[current_syll].append(ph) #Phonemes are added to a syllable as long as no new vowel is found in the list of phonemes.
            if any(ph.startswith(v) for v in VOWELS): # As soon as there's a vowel, it means we're moving on to another syllable.
                current_syll += 1 if current_syll +1 < len(toke_syll) else 0
                vowel_count += 1 # ccount

        stress_per_syll = []
        vowel_counter = 0
        # then we create a list of 0/1s, the size of which is the number of syllables, with 0 by default (no accent) and 1 as soon as we find the syllable containing the vowel accompanied by a "1"
        for syll in phonemes_per_syll:
            stressed = 0  # by default
            for ph in syll:
                if any(ph.startswith(v) for v in VOWELS):
                    if ph[-1] == '1':  
                        stressed = 1
                    vowel_counter += 1
                    break
            stress_per_syll.append(stressed)

        for index, syll in enumerate(toke_syll) :
            syllables_list.append((syll, token, index, stress_per_syll[index], monosyllabic))
            
    return syllables_list

## 5. Data Preprocessing

### 5.1 Algorithm

Our idea is to transform each line into a dataset where:
- 1 syllable = 1 line (of data) ;
- then the columns are the features per syllable (lexical stress, post-tag if monosyllabic word, position in the verse, etc.)

In [ ]:
def preprocess_line(line) :
    # tokenisation 
    line = line.lower()
    tokens = nltk.tokenize.word_tokenize(line, language="english")
    tokens = [token for token in tokens if not is_punct(token)]
    tags = dict(nltk.pos_tag(tokens)) # pos tagging 

    syllables_list = lexical_stress_tokens(tokens, tags=tags) # see previosu functions

    syllables_dict = {index:syllable_tuple  for index, syllable_tuple in enumerate(syllables_list)}
    df_line = pd.DataFrame.from_dict(syllables_dict, orient='index', columns=["syllable", "word", "position_in_word", "lexical_stress", "monosyllabic"])

    # a few more (basic) features 
    first = []
    end = []
    for i in range(len(syllables_list)) :
        if i == 0 :
            first.append(1.0)
            end.append(0.0)
        elif i == len(syllables_list) -1:
            first.append(0.0)
            end.append(1.0)
        else :
            first.append(0.0)
            end.append(0.0)
            
    df_line["first"] = first
    df_line["end"] = end
    df_line["pos"] = df_line["word"].map(tags)
    # Special cases for monosyllabic words to verify intuition about function words (unstressed in poetry)
    df_line["determiner"] = ((df_line["pos"].isin(["DT", "PDT", "WDT"])) & (df_line["monosyllabic"] == 1)).astype(int)
    df_line["preposition"] = ((df_line["pos"] == "IN") & (df_line["monosyllabic"] == 1)).astype(int)
    df_line["to"] = ((df_line["pos"] == "TO") & (df_line["monosyllabic"] == 1)).astype(int)
    df_line["interjection"] = ((df_line["pos"] == "UH") & (df_line["monosyllabic"] == 1)).astype(int)
    df_line["particle"] = ((df_line["pos"] == "RP") & (df_line["monosyllabic"] == 1)).astype(int)

    return df_line
    

In [98]:
a = preprocess_line(line=line)

In [99]:
a

,syllable,word,position_in_word,lexical_stress,monosyllabic,first,end,pos,determiner,preposition,to,interjection,particle
0,what,what,0,1,1,1.0,0.0,WP,0,0,0,0,0
1,was,was,0,1,1,0.0,0.0,VBD,0,0,0,0,0
2,he,he,0,1,1,0.0,0.0,PRP,0,0,0,0,0
3,do,doing,0,1,0,0.0,0.0,VBG,0,0,0,0,0
4,ing,doing,1,0,0,0.0,0.0,VBG,0,0,0,0,0
5,the,the,0,0,1,0.0,0.0,DT,1,0,0,0,0
6,great,great,0,1,1,0.0,0.0,JJ,0,0,0,0,0
7,god,god,0,1,1,0.0,0.0,NN,0,0,0,0,0
8,pan,pan,0,1,1,0.0,1.0,NN,0,0,0,0,0


## 5.2 Applying the algorithm to our dataset

In [ ]:
df = pd.read_csv(r"forbetterverse.csv")

In [ ]:
# in order to add the gold standard
def to_binary(sequence):
    n_seq = []
    for i in sequence:
        if i == "+":
            n_seq.append(torch.tensor(1.0))
        else :
            n_seq.append(torch.tensor(0.0))
    return n_seq
to_binary('+--+--+++')

[tensor(1.),
 tensor(0.),
 tensor(0.),
 tensor(1.),
 tensor(0.),
 tensor(0.),
 tensor(1.),
 tensor(1.),
 tensor(1.)]

In [102]:
len(df["real"][0])

9

In [103]:
len(to_binary(df["real"][0]))

9

In [104]:
df["line"]

0                What was he doing, the great god Pan,
1                      Down in the reeds by the river?
2                   Spreading ruin and scattering ban,
3         Splashing and paddling with hoofs of a goat,
4                And breaking the golden lilies afloat
                             ...                      
1477      And get knocked on the head for his labours,
1478     To do good to mankind is the chivalrous plan,
1479                  And is always as nobly requited;
1480         Then battle for freedom wherever you can,
1481    And if not shot or hanged you'll get knighted.
Name: line, Length: 1482, dtype: object

In [105]:
preprocess_line(df["line"][0]).shape[0]

9

In [107]:
dataframes = []
for index, row in tqdm(df.iterrows()):
    try :
        real = to_binary(row["real"])
        a = preprocess_line(line=row["line"])
        if a.shape[0] == len(real) : 
            dataframes.append((a, real))
    except :
        KeyError


755it [00:02, 365.62it/s]

ok


1482it [00:04, 338.65it/s]


In [106]:
len(dataframes)

427

In [108]:
len(dataframes) # 1400/1482

759

In [19]:
dataframes[0][1]

[tensor(1.),
 tensor(0.),
 tensor(0.),
 tensor(1.),
 tensor(0.),
 tensor(0.),
 tensor(1.),
 tensor(1.),
 tensor(1.)]

# 5. Machine learning model

In [109]:
class syllable_labelling(nn.Module):

    def __init__(self):
        super(syllable_labelling, self).__init__()
        #self.x1 = torch.tensor(syllable["lexical_stress"])
        #self.x2 = torch.tensor(syllable["position_in_word"])
        #self.x3 = torch.tensor(syllable["determiner"])
        #self.x4 = torch.tensor(syllable["preposition"])
        #self.x5 = torch.tensor(syllable["to"])
        #self.x6 = torch.tensor(syllable["interjection"])
        #self.x7 = torch.tensor(syllable["particle"])
        self.linear = nn.Linear(7, 1)
    def forward(self, X):
        prediction = torch.sigmoid(self.linear(X))
        return prediction

In [110]:
a = syllable_labelling()
a

syllable_labelling(
  (linear): Linear(in_features=7, out_features=1, bias=True)
)

In [47]:
class line_labelling(nn.Module):

    def __init__(self):
        super(line_labelling, self).__init__()
        self.syllable_labelling = syllable_labelling()
    def forward(self, line):
        features = []
        for _, row in line.iterrows():
            features.append([row["lexical_stress"], row["position_in_word"], row["determiner"], row["preposition"], row["to"], row["interjection"], row["particle"]])
        X = torch.tensor(features, dtype=torch.float32)
        return self.syllable_labelling(X).squeeze()

## 6. Improved model ?

In [119]:
FEATURE_NAMES = [
    "lexical_stress",
    "position_in_word",
    "determiner",
    "preposition",
    "to",
    "interjection",
    "particle"
]

In [120]:
def build_context_features(df):
    features = df[FEATURE_NAMES].values
    n = len(features)
    
    context_features = []
    
    for i in range(n):
        prev_feat = features[i-1] if i > 0 else np.zeros_like(features[0])
        curr_feat = features[i]
        next_feat = features[i+1] if i < n-1 else np.zeros_like(features[0])
        
        combined = np.concatenate([prev_feat, curr_feat, next_feat])
        context_features.append(combined)
    
    return np.array(context_features)

In [ ]:
class SyllableModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(21, 1) 

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

In [130]:
class LineModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.syllable_model = SyllableModel()

    def forward(self, df):
        x = build_context_features(df)
        x = torch.tensor(x, dtype=torch.float32)

        preds = self.syllable_model(x)
        return preds.view(-1)
        

In [111]:
round(len(dataframes) * 0.7)

531

In [112]:
759 * 0.8

607.2

In [26]:
a = [1,2,3,4,4]
a[:2]

[1, 2]

In [43]:
train, dev, test = dataframes[:981], dataframes[981:1192], dataframes[1192:]

In [124]:
train = dataframes[:600]

In [114]:
def hamming_distance(seq1, seq2):
    assert len(seq1) == len(seq2)
    return sum(el1 != el2 for el1, el2 in zip(seq1, seq2))/len(seq1)

In [ ]:
def train_loop(data, num_epochs, learning_rate):
    #model = line_labelling()
    model = LineModel()
    optimizer = optim.Adam(model.parameters(),lr=learning_rate)
    criterion = nn.BCELoss()
    for epoch in range(num_epochs):
        model.train()
        for d in data:
            prediction = model(d[0])
            #print(prediction)
            loss = criterion(prediction, torch.tensor(d[1], dtype=torch.float32)) 
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        model.eval()
        all_preds = []
        for d in data:
            preds = model(d[0]).detach().numpy()
            preds_round = [int(p > 0.5) for p in preds]
            all_preds.append((preds_round, d[1]))
        # mean
        if epoch % 10 == 0 :
            accuracy = np.mean([np.mean([p == g for p, g in zip(pred, gold)]) for pred, gold in all_preds])
            # hamming
            #hammings = np.mean([hamming_distance(pred, gold) for pred, gold in all_preds])
            print(f'Training complete. Pseudo-Accuracy={accuracy*100:.2f}%')
            #print(f'Training complete. Mean Hamming Distance={hammings*100:.2f}%')
    return model

In [137]:
train_loop(train, 150, 0.001)

Training complete. Pseudo-Accuracy=72.50%
Training complete. Pseudo-Accuracy=75.59%
Training complete. Pseudo-Accuracy=75.92%
Training complete. Pseudo-Accuracy=75.98%
Training complete. Pseudo-Accuracy=76.00%
Training complete. Pseudo-Accuracy=76.00%
Training complete. Pseudo-Accuracy=76.00%
Training complete. Pseudo-Accuracy=76.02%
Training complete. Pseudo-Accuracy=76.02%
Training complete. Pseudo-Accuracy=76.02%
Training complete. Pseudo-Accuracy=76.02%
Training complete. Pseudo-Accuracy=76.02%
Training complete. Pseudo-Accuracy=76.02%
Training complete. Pseudo-Accuracy=76.02%
Training complete. Pseudo-Accuracy=76.02%


LineModel(
  (syllable_model): SyllableModel(
    (linear): Linear(in_features=21, out_features=1, bias=True)
  )
)